# Player and Goalkeeper Ability

This script is my attempt at feature engineering to find different, reliable ways of quantifying the player's scoring ability and the goalkeeper's saving abilities for every shot taken.

This is to perform a shot-level analysis of shooter and goalkeeper effects to ultimately answer the question:
    Does player finishing ability and goalkeeper ability improve shot-level xG beyond geometry/context?

*Uncomment commented cells to stream and get full dataset if you don't have that already*

In [19]:
import pandas as pd

shots = pd.read_csv("~/Desktop/Football_Stats/xG/datasets/processed/Big5_shots.csv")
shots.head()

,location,player,player_id,position,shot_aerial_won,shot_first_time,shot_statsbomb_xg,under_pressure,shot_open_goal,shot_follows_dribble,...,shot_technique_Overhead Kick,shot_technique_Volley,shot_type_Corner,shot_type_Free Kick,shot_type_Open Play,shot_type_Penalty,x,y,distance,angle
0,"[94.5, 42.9]",Aaron Ramsey,3517.0,Right Wing,0,1,0.038832,0,0,0,...,0,0,0,0,1,0,94.5,42.9,25.664372,17.611035
1,"[93.5, 48.4]",Francesc Fàbregas i Soler,3478.0,Right Defensive Midfield,0,0,0.031541,1,0,0,...,0,0,0,0,1,0,93.5,48.4,27.799460,15.648789
2,"[89.8, 22.3]",Alexis Alejandro Sánchez Sánchez,3385.0,Left Wing,0,0,0.006660,1,0,0,...,0,0,0,0,1,0,89.8,22.3,35.004714,11.297814
3,"[98.2, 26.0]",Diego da Silva Costa,5198.0,Center Forward,0,0,0.022902,0,0,0,...,0,0,0,0,1,0,98.2,26.0,25.908300,14.904419
4,"[105.7, 24.2]",Theo Walcott,3668.0,Center Forward,0,0,0.049741,0,0,0,...,0,0,0,0,1,0,105.7,24.2,21.310326,14.633756


In [20]:
shots.player.value_counts().sort_values()

player
Mirko Valdifiori                         1
Gary Hooper                              1
Eunan O'Kane                             1
Marvin Emnes                             1
Seydou Doumbia                           1
                                      ... 
Zlatan Ibrahimović                     148
Harry Kane                             158
Lionel Andrés Messi Cuccittini         158
Gonzalo Gerardo Higuaín                182
Cristiano Ronaldo dos Santos Aveiro    228
Name: count, Length: 1991, dtype: int64

Classical xG mostly asks:
    Given the shot situation, how likely is this shot to become a goal?

From further investigation, I think I will eventually remove penalties from the dataset. Although the models learn to differentiate scenarios(open player, set pieces and penalties), penalties present much more "constant" features than other shot-types.

They involve fixed geometry, nearly fixed angle, nearly fixed distance, and are pychologically and tactically unique. Not to mention, on further data investigation, the xG of penalties is fixed between ~ 0.75 - 0.80. 

I will perform analysis on the full data, then either seperate the model into penalties and non-penalties and perform analysis on the different datasets, or Keep penalties in dataset with penalty indicator feature.


In [21]:
shots.columns

Index(['location', 'player', 'player_id', 'position', 'shot_aerial_won',
       'shot_first_time', 'shot_statsbomb_xg', 'under_pressure',
       'shot_open_goal', 'shot_follows_dribble', 'goalkeeper_success_out',
       'half_end_early_video_end', 'goal', 'shot_body_part_Head',
       'shot_body_part_Left Foot', 'shot_body_part_Other',
       'shot_body_part_Right Foot', 'shot_technique_Backheel',
       'shot_technique_Diving Header', 'shot_technique_Half Volley',
       'shot_technique_Lob', 'shot_technique_Normal',
       'shot_technique_Overhead Kick', 'shot_technique_Volley',
       'shot_type_Corner', 'shot_type_Free Kick', 'shot_type_Open Play',
       'shot_type_Penalty', 'x', 'y', 'distance', 'angle'],
      dtype='object')

For the player/goalkeeper ability ratings, I will use a dataset, scraped by (https://www.kaggle.com/datasets/yarknyorulmaz/fifa-index-player-ratings-dataset-epl-v16-v24), which contain detailed player statistics and attributes for football players as featured in the FIFAIndex database, which is a widely recognized source of football player data used for simulation and analysis purposes. 

In [ ]:
# %pip install "absl-py>=0.4" "werkzeug>=1.0.1" "protobuf>=3.19.6,<4.24" kagglehub
# %pip install kagglehub[pandas-datasets]

In [ ]:
# import kagglehub
# import pandas as pd
# import os

# dataset = "yarknyorulmaz/fifa-index-player-ratings-dataset-epl-v16-v24"

# # Download dataset locally
# path = kagglehub.dataset_download(dataset)

# print("Dataset downloaded to:", path)
# print(os.listdir(path))

In [30]:
df = pd.read_csv("~/Desktop/Football_Stats/xG/datasets/raw/players_16.csv")
df.head()

/var/folders/lz/th4y3_7n34lfmhmw0b_t5y0r0000gn/T/ipykernel_81789/3489828024.py:1: DtypeWarning: Columns (104) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("~/Desktop/Football_Stats/xG/datasets/raw/players_16.csv")


,sofifa_id,player_url,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,...,lcb,cb,rcb,rb,gk,player_face_url,club_logo_url,club_flag_url,nation_logo_url,nation_flag_url
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,"RW, CF",94,95,111000000.0,550000.0,28,...,44+3,44+3,44+3,57+3,19+3,https://cdn.sofifa.net/players/158/023/16_120.png,https://cdn.sofifa.net/teams/241/60.png,https://cdn.sofifa.net/flags/es.png,https://cdn.sofifa.net/teams/1369/60.png,https://cdn.sofifa.net/flags/ar.png
1,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"LW, LM",93,93,85500000.0,475000.0,30,...,52+3,52+3,52+3,60+3,20+3,https://cdn.sofifa.net/players/020/801/16_120.png,https://cdn.sofifa.net/teams/243/60.png,https://cdn.sofifa.net/flags/es.png,https://cdn.sofifa.net/teams/1354/60.png,https://cdn.sofifa.net/flags/pt.png
2,9014,https://sofifa.com/player/9014/arjen-robben/16...,A. Robben,Arjen Robben,"RM, LM, RW",90,90,56000000.0,250000.0,31,...,47+3,47+3,47+3,59+3,19+3,https://cdn.sofifa.net/players/009/014/16_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/105035/60.png,https://cdn.sofifa.net/flags/nl.png
3,167495,https://sofifa.com/player/167495/manuel-neuer/...,M. Neuer,Manuel Peter Neuer,GK,90,90,58000000.0,250000.0,29,...,33+3,33+3,33+3,33+3,87+3,https://cdn.sofifa.net/players/167/495/16_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/1337/60.png,https://cdn.sofifa.net/flags/de.png
4,176580,https://sofifa.com/player/176580/luis-suarez/1...,L. Suárez,Luis Alberto Suárez Díaz,ST,90,90,69000000.0,300000.0,28,...,58+3,58+3,58+3,64+3,37+3,https://cdn.sofifa.net/players/176/580/16_120.png,https://cdn.sofifa.net/teams/241/60.png,https://cdn.sofifa.net/flags/es.png,NaN,https://cdn.sofifa.net/flags/uy.png


In [31]:
# df = df[df['Version'] =='fifa16']
# df.head()

In [32]:
df.columns

Index(['sofifa_id', 'player_url', 'short_name', 'long_name',
       'player_positions', 'overall', 'potential', 'value_eur', 'wage_eur',
       'age',
       ...
       'lcb', 'cb', 'rcb', 'rb', 'gk', 'player_face_url', 'club_logo_url',
       'club_flag_url', 'nation_logo_url', 'nation_flag_url'],
      dtype='object', length=110)

In [ ]:
# cols = ["Player Name", "Att. Position", "Finishing", "Shot Power", "Long Shots", "Volleys", "Penalties", "Heading"]
cols = ['sofifa_id', 'player_url', 'short_name', 'long_name', 'shooting', 'attacking_heading_accuracy', 'preferred_foot', 'weak_foot']
# I am including "attacking_heading_accuracy" since some scoring contexts are aerial won, 
# meaning the player's heading accuracy is necessary for accurately capturing their scoring ability in aerial situations
df = df[cols].copy()
df['Shooting'] = (df['shooting'] + df['attacking_heading_accuracy']) / 2
df.drop(columns=['shooting', 'attacking_heading_accuracy'], inplace=True)
df.head()

,sofifa_id,player_url,short_name,long_name,Shooting
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,79.5
1,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,89.5
2,9014,https://sofifa.com/player/9014/arjen-robben/16...,A. Robben,Arjen Robben,68.5
3,167495,https://sofifa.com/player/167495/manuel-neuer/...,M. Neuer,Manuel Peter Neuer,NaN
4,176580,https://sofifa.com/player/176580/luis-suarez/1...,L. Suárez,Luis Alberto Suárez Díaz,82.5


In [ ]:
# df["Shooting"] = (df["Att. Position"] + df["Finishing"] + df["Shot Power"] + df["Long Shots"] + df["Volleys"] + df["Penalties"] + df["Heading"]) / 7
# I am including "Heading" since some scoring contexts are aerial won, meaning the player's heading accuracy is necessary
# df["Saving"] = (df["GK Diving"] + df["GK Handling"] + df["GK Positioning"] + df["GK Reflexes"])/4 
# I am not including "GK Kicking" since it doesn't give a measure of saving ability

# df = df[["Player Name", "Shooting", "Saving"]].copy()

# df["Source"] = "yarknyorulmaz/fifa-index-player-ratings-dataset-epl-v16-v24"

,sofifa_id,player_url,short_name,long_name,Shooting,Source
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,79.5,SoFIFA Platform
1,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,89.5,SoFIFA Platform
2,9014,https://sofifa.com/player/9014/arjen-robben/16...,A. Robben,Arjen Robben,68.5,SoFIFA Platform
3,167495,https://sofifa.com/player/167495/manuel-neuer/...,M. Neuer,Manuel Peter Neuer,NaN,SoFIFA Platform
4,176580,https://sofifa.com/player/176580/luis-suarez/1...,L. Suárez,Luis Alberto Suárez Díaz,82.5,SoFIFA Platform


In [ ]:
# for now since we are focusing on shooting ability
df = df.dropna(subset=['Shooting'])

In [ ]:
df["Source"] = "SoFIFA Platform"
df.head()

In [35]:
df["short_name"].value_counts()

short_name
J. Rodríguez      10
J. García          7
M. Díaz            6
R. Williams        6
A. Traoré          6
                  ..
P. Flo             1
S. Siani           1
M. Komorowski      1
Lee Seung Hyun     1
C. Shephard        1
Name: count, Length: 14747, dtype: int64

SUPPLIMENTAL PLAYERS

In [40]:
# Players in the game with ratings, that are featured in the statsbomb dataset, but not in the fifa-player-ratings-dataset. 
# This is due to transfer during the season.

supplemental_players = pd.DataFrame([
    {
        "Player Name": "Adam Johnson",
        "statsbomb_name": "Adam Johnson",
        "Att. Position": 77,
        "Finishing": 75,
        "Shot Power": 74,
        "Long Shots": 78,
        "Volleys": 67,
        "Penalties": 68,
        "Heading": 44,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Steven Fletcher",
        "statsbomb_name": "Steven Fletcher",
        "Att. Position": 78,
        "Finishing": 80,
        "Shot Power": 76,
        "Long Shots": 67,
        "Volleys": 66,
        "Penalties": 71,
        "Heading": 86,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Mikel Arteta Amatriain",
        "statsbomb_name": "Mikel Arteta Amatriain",
        "Att. Position": 70,
        "Finishing": 75,
        "Shot Power": 74,
        "Long Shots": 78,
        "Volleys": 67,
        "Penalties": 68,
        "Heading": 44,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Ramires Santos do Nascimen.",
        "statsbomb_name": "Ramires Santos do Nascimento",
        "Att. Position": 77,
        "Finishing": 68,
        "Shot Power": 77,
        "Long Shots": 72,
        "Volleys": 75,
        "Penalties": 85,
        "Heading": 64,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Mauro Zárate",
        "statsbomb_name": "Mauro Matías Zárate",
        "Att. Position": 79,
        "Finishing": 75,
        "Shot Power": 80,
        "Long Shots": 82,
        "Volleys": 78,
        "Penalties": 70,
        "Heading": 58,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Éder Macedo Lopes",
        "statsbomb_name": "Éderzito António Macedo Lopes",
        "Att. Position": 78,
        "Finishing": 73,
        "Shot Power": 79,
        "Long Shots": 69,
        "Volleys": 75,
        "Penalties": 73,
        "Heading": 7,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Bradley Johnson",
        "statsbomb_name": "Bradley Johnson",
        "Att. Position": 72,
        "Finishing": 70,
        "Shot Power": 91,
        "Long Shots": 78,
        "Volleys": 68,
        "Penalties": 72,
        "Heading": 76,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Florian Thauvin",
        "statsbomb_name": "Florian Thauvin",
        "Att. Position": 74,
        "Finishing": 70,
        "Shot Power": 76,
        "Long Shots": 74,
        "Volleys": 64,
        "Penalties": 57,
        "Heading": 66,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Nikica Jelavić",
        "statsbomb_name": "Nikica Jelavić",
        "Att. Position": 75,
        "Finishing": 77,
        "Shot Power": 74,
        "Long Shots": 70,
        "Volleys": 69,
        "Penalties": 81,
        "Heading": 77,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Brede Hangeland",
        "statsbomb_name": "Brede Hangeland",
        "Att. Position": 19,
        "Finishing": 34,
        "Shot Power": 63,
        "Long Shots": 39,
        "Volleys": 38,
        "Penalties": 44,
        "Heading": 80,
        "Source": "manual_fifa_index"
    }, # Damn this nigga trash fr fr
    {
        "Player Name": "Javier Hernández",
        "statsbomb_name": "Javier Hernández Balcázar",
        "Att. Position": 88,
        "Finishing": 89,
        "Shot Power": 75,
        "Long Shots": 69,
        "Volleys": 79,
        "Penalties": 80,
        "Heading": 82,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Miguel Layún",
        "statsbomb_name": "Miguel Arturo Layún Prado",
        "Att. Position": 76,
        "Finishing": 68,
        "Shot Power": 80,
        "Long Shots": 77,
        "Volleys": 68,
        "Penalties": 75,
        "Heading": 56,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Gary Hooper",
        "statsbomb_name": "Gary Hooper",
        "Att. Position": 75,
        "Finishing": 77,
        "Shot Power": 77,
        "Long Shots": 67,
        "Volleys": 67,
        "Penalties": 72,
        "Heading": 74,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Modibo Maïga",
        "statsbomb_name": "Modibo Maïga",
        "Att. Position": 64,
        "Finishing": 68,
        "Shot Power": 68,
        "Long Shots": 62,
        "Volleys": 65,
        "Penalties": 57,
        "Heading": 67,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Juan Cuadrado",
        "statsbomb_name": "Juan Guillermo Cuadrado Bello",
        "Att. Position": 77,
        "Finishing": 75,
        "Shot Power": 74,
        "Long Shots": 78,
        "Volleys": 67,
        "Penalties": 68,
        "Heading": 44,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Alessandro Diamanti",
        "statsbomb_name": "Alessandro Diamanti",
        "Att. Position": 78,
        "Finishing": 71,
        "Shot Power": 83,
        "Long Shots": 80,
        "Volleys": 73,
        "Penalties": 65,
        "Heading": 58,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Steven Taylor",
        "statsbomb_name": "Steven Taylor",
        "Att. Position": 20,
        "Finishing": 32,
        "Shot Power": 43,
        "Long Shots": 24,
        "Volleys": 59,
        "Penalties": 46,
        "Heading": 74,
        "Source": "manual_fifa_index"
    },
    {
        "Player Name": "Mathieu Debuchy",
        "statsbomb_name": "Mathieu Debuchy",
        "Att. Position": 64,
        "Finishing": 59,
        "Shot Power": 74,
        "Long Shots": 68,
        "Volleys": 62,
        "Penalties": 66,
        "Heading": 75,
        "Source": "manual_fifa_index"
    },

    {
        "Player Name": "Yann Kermorgant",
        "statsbomb_name": "Yann Kermorgant",
        "Att. Position": 66,
        "Finishing": 73,
        "Shot Power": 75,
        "Long Shots": 70,
        "Volleys": 73,
        "Penalties": 71,
        "Heading": 78,
        "Source": "manual_fifa_index"
        }

])

# "source": "manual_fifa_index"
# "rating_version": "FIFA 16"
# "notes": "Transferred out / absent from EPL snapshot"


# Mapping-Table Pipeline

Normalize Names

In [38]:
import unicodedata
from rapidfuzz import process, fuzz


def normalize_name(name):
    name = str(name).lower().strip()
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("utf-8")
    name = name.replace(".", "")
    name = name.replace("-", " ")
    # name = name.replace("’", "'").replace("‘", "'").replace("`", "'").replace("´", "'")
    # name = name.replace("''", "'") 
    name = " ".join(name.split()) # normalize spacing

    return name

# def normalize_name(name):
#     name = str(name).lower().strip()

#     # Standardize apostrophes first
#     name = (
#         name.replace("’", "'")
#             .replace("‘", "'")
#             .replace("`", "'")
#             .replace("´", "'")
#             .replace("''", "'")
#     )

#     # Remove accents
#     name = unicodedata.normalize("NFKD", name)
#     name = name.encode("ascii", "ignore").decode("utf-8")

#     # Standardize punctuation/spacing
#     name = name.replace(".", "")
#     name = name.replace("-", " ")
#     name = " ".join(name.split())

#     return name

In [41]:
# Normalize the supplemental players. Keep the StatsBomb alias so these rows can
# match the shot data even when the FIFA display name is shortened/truncated.
supplemental_players["statsbomb_name_norm"] = supplemental_players["statsbomb_name"].apply(normalize_name)

base_fifa_players = df.copy()
base_fifa_players["statsbomb_name"] = pd.NA
base_fifa_players["statsbomb_name_norm"] = pd.NA

supplemental_players_for_match = supplemental_players[[
    "Player Name", "statsbomb_name", "statsbomb_name_norm", "Att. Position", "Finishing", "Shot Power", "Long Shots", 
    "Volleys", "Penalties", "Heading", "Source"]].copy()

# supplemental_players_for_match = supplemental_players[[
#     "Player Name",
#     "statsbomb_name",
#     "statsbomb_name_norm",
#     "Shooting",
#     "Saving",
#     "Source",
# ]].copy()

df_extended = pd.concat(
    [base_fifa_players, supplemental_players_for_match],
    ignore_index=True
)

df_extended["fifa_name_norm"] = df_extended["Player Name"].apply(normalize_name)
df_extended["match_name_norm"] = df_extended["statsbomb_name_norm"].fillna(df_extended["fifa_name_norm"])
df_extended.head()


,sofifa_id,player_url,short_name,long_name,Shooting,Source,statsbomb_name,statsbomb_name_norm,Player Name,Att. Position,Finishing,Shot Power,Long Shots,Volleys,Penalties,Heading,fifa_name_norm,match_name_norm
0,158023.0,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,79.5,SoFIFA Platform,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,nan
1,20801.0,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,89.5,SoFIFA Platform,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,nan
2,9014.0,https://sofifa.com/player/9014/arjen-robben/16...,A. Robben,Arjen Robben,68.5,SoFIFA Platform,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,nan
3,167495.0,https://sofifa.com/player/167495/manuel-neuer/...,M. Neuer,Manuel Peter Neuer,NaN,SoFIFA Platform,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,nan
4,176580.0,https://sofifa.com/player/176580/luis-suarez/1...,L. Suárez,Luis Alberto Suárez Díaz,82.5,SoFIFA Platform,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,nan


Create unique player tables

In [42]:
# StatsBomb unique shooters
sb_players = shots[["player_id", "player"]].drop_duplicates().copy()
sb_players["statsbomb_name_norm"] = sb_players["player"].apply(normalize_name)

# FIFA unique players. match_name_norm is the name used for matching:
# regular FIFA rows use Player Name; supplemental rows use statsbomb_name.
fifa_players = df_extended[[
    "Player Name", "statsbomb_name", "Att. Position", "Finishing", "Shot Power", "Long Shots", 
    "Volleys", "Penalties", "Heading", "Source", "fifa_name_norm", "match_name_norm"]].drop_duplicates().copy()


Extracting Matches

In [43]:
exact_matches = sb_players.merge(
    fifa_players,
    left_on="statsbomb_name_norm",
    right_on="match_name_norm",
    how="inner"
)

exact_matches["match_method"] = "exact"
exact_matches["match_score"] = 100
exact_matches.head()


,player_id,player,statsbomb_name_norm,Player Name,statsbomb_name,Att. Position,Finishing,Shot Power,Long Shots,Volleys,Penalties,Heading,Source,fifa_name_norm,match_name_norm,match_method,match_score
0,28100.0,Ramires Santos do Nascimento,ramires santos do nascimento,Ramires Santos do Nascimen.,Ramires Santos do Nascimento,77.0,68.0,77.0,72.0,75.0,85.0,64.0,manual_fifa_index,ramires santos do nascimen,ramires santos do nascimento,exact,100
1,9554.0,Steven Fletcher,steven fletcher,Steven Fletcher,Steven Fletcher,78.0,80.0,76.0,67.0,66.0,71.0,86.0,manual_fifa_index,steven fletcher,steven fletcher,exact,100
2,15996.0,Éderzito António Macedo Lopes,ederzito antonio macedo lopes,Éder Macedo Lopes,Éderzito António Macedo Lopes,78.0,73.0,79.0,69.0,75.0,73.0,7.0,manual_fifa_index,eder macedo lopes,ederzito antonio macedo lopes,exact,100
3,42703.0,Mikel Arteta Amatriain,mikel arteta amatriain,Mikel Arteta Amatriain,Mikel Arteta Amatriain,70.0,75.0,74.0,78.0,67.0,68.0,44.0,manual_fifa_index,mikel arteta amatriain,mikel arteta amatriain,exact,100
4,42902.0,Adam Johnson,adam johnson,Adam Johnson,Adam Johnson,77.0,75.0,74.0,78.0,67.0,68.0,44.0,manual_fifa_index,adam johnson,adam johnson,exact,100


Finding Unmatched names

In [44]:
matched_sb_ids = set(exact_matches["player_id"])

unmatched_sb = sb_players[
    ~sb_players["player_id"].isin(matched_sb_ids)
].copy()

print("Unmatched StatsBomb players:", len(unmatched_sb))
unmatched_sb.to_csv("~/Desktop/Football_Stats/xG/datasets/Big5_unmatched_sb_players.csv", index=True)

Unmatched StatsBomb players: 1972


Generate fuzzy candidates

In [45]:
fifa_name_list = fifa_players["match_name_norm"].tolist()

candidate_rows = []

for _, row in unmatched_sb.iterrows():
    candidates = process.extract(
        row["statsbomb_name_norm"],
        fifa_name_list,
        scorer=fuzz.WRatio,
        limit=5
    )

    for candidate_name, score, idx in candidates:
        fifa_row = fifa_players.iloc[idx]

        candidate_rows.append({
            "player_id": row["player_id"],
            "statsbomb_name": row["player"],
            "statsbomb_name_norm": row["statsbomb_name_norm"],
            "fifa_player_name": fifa_row["Player Name"],
            "fifa_name_norm": fifa_row["fifa_name_norm"],
            "match_name_norm": fifa_row["match_name_norm"],
            "Source": fifa_row["Source"],
            "match_method": "fuzzy",
            "match_score": score
        })

fuzzy_candidates = pd.DataFrame(candidate_rows)

fuzzy_candidates.sort_values(
    ["statsbomb_name", "match_score"],
    ascending=[True, False]
).head(100)


,player_id,statsbomb_name,statsbomb_name_norm,fifa_player_name,fifa_name_norm,match_name_norm,Source,match_method,match_score
1370,3062.0,Aaron Cresswell,aaron cresswell,Juan Cuadrado,juan cuadrado,juan guillermo cuadrado bello,manual_fifa_index,fuzzy,46.666667
1371,3062.0,Aaron Cresswell,aaron cresswell,Mikel Arteta Amatriain,mikel arteta amatriain,mikel arteta amatriain,manual_fifa_index,fuzzy,46.216216
1372,3062.0,Aaron Cresswell,aaron cresswell,NaN,nan,nan,SoFIFA Platform,fuzzy,45.000000
1373,3062.0,Aaron Cresswell,aaron cresswell,Éder Macedo Lopes,eder macedo lopes,ederzito antonio macedo lopes,manual_fifa_index,fuzzy,39.461538
1374,3062.0,Aaron Cresswell,aaron cresswell,Mauro Zárate,mauro zarate,mauro matias zarate,manual_fifa_index,fuzzy,39.117647
...,...,...,...,...,...,...,...,...,...
330,4090.0,Adam David Lallana,adam david lallana,Adam Johnson,adam johnson,adam johnson,manual_fifa_index,fuzzy,85.500000
331,4090.0,Adam David Lallana,adam david lallana,NaN,nan,nan,SoFIFA Platform,fuzzy,72.000000
332,4090.0,Adam David Lallana,adam david lallana,Alessandro Diamanti,alessandro diamanti,alessandro diamanti,manual_fifa_index,fuzzy,43.243243
333,4090.0,Adam David Lallana,adam david lallana,Juan Cuadrado,juan cuadrado,juan guillermo cuadrado bello,manual_fifa_index,fuzzy,42.000000


In [46]:
fuzzy_candidates.sort_values(
    ["statsbomb_name", "match_score"],
    ascending=[True, False]
).head()

,player_id,statsbomb_name,statsbomb_name_norm,fifa_player_name,fifa_name_norm,match_name_norm,Source,match_method,match_score
1370,3062.0,Aaron Cresswell,aaron cresswell,Juan Cuadrado,juan cuadrado,juan guillermo cuadrado bello,manual_fifa_index,fuzzy,46.666667
1371,3062.0,Aaron Cresswell,aaron cresswell,Mikel Arteta Amatriain,mikel arteta amatriain,mikel arteta amatriain,manual_fifa_index,fuzzy,46.216216
1372,3062.0,Aaron Cresswell,aaron cresswell,NaN,nan,nan,SoFIFA Platform,fuzzy,45.000000
1373,3062.0,Aaron Cresswell,aaron cresswell,Éder Macedo Lopes,eder macedo lopes,ederzito antonio macedo lopes,manual_fifa_index,fuzzy,39.461538
1374,3062.0,Aaron Cresswell,aaron cresswell,Mauro Zárate,mauro zarate,mauro matias zarate,manual_fifa_index,fuzzy,39.117647


In [ ]:
# fuzzy_candidates = fuzzy_candidates[fuzzy_candidates['match_score'] >= 65.0]
# fuzzy_candidates.shape


In [ ]:
# Saving for manual review

# fuzzy_candidates.to_csv("fuzzy_player_candidates.csv", index=False)

While manually dealing with the mismatches, some mismatches were kept since they're replacements were not in the file. So I will use the dictionary below to handle those.

In [47]:
manual_overrides = {
    # "statsbomb normalized name": "fifa normalized name"
    # "mikel arteta amatriain": "mikel arteta amatriain", # **
    "fernando luiz rosa": "fernandinho",
    "fernando luiz roza": "fernandinho",
    "jose leonardo ulloa": "leonardo ulloa",
    "n''golo kante": "n'golo kante",
    "n'golo kante": "n'golo kante",
    "yann gerard m''vila": "yann m'vila",
    "yann gerard m'vila": "yann m'vila",
    "jordi gomez garcia penche": "jordi gomez",
    "jose salomon rondon gimenez": "salomon rondon",
    "claudio ariel yacob": "claudio yacob",
    "idrissa gana gueye": "idrissa gueye",
    "siem stefan de jong": "siem de jong",
    "carlos alberto sanchez moreno": "carlos sanchez",
    "ayoze perez gutierrez": "ayoze perez",
    "el hadji baye oumar niasse": "oumar niasse",
    "pape n''diaye souare": "pape souare",
    "odion jude ighalo": "odion ighalo",
    "chung yong lee": "lee chung yong",
    "gabriel armando de abreu": "gabriel",
    "alexis alejandro sanchez sanchez": "alexis sanchez",
    "hector bellerin moruno": "hector bellerin",
    "mohamed naser elsayed elneny": "mohamed elneny",
    "santiago cazorla gonzalez": "santi cazorla",
    "oluwaseyi babajide ojo": "sheyi ojo",
    "christian benteke liolo": "christian benteke",
    "christian dannemann eriksen": "christian eriksen",
    "bamidele alli": "dele alli",
    "sergio leonel aguero del castillo": "sergio aguero",
    "nicolas hernan otamendi": "nicolas otamendi",
    "gnegneri yaya toure": "yaya toure",
    "fernando luiz roza": "fernandinho",
    "wilfried guemiand bony": "wilfried bony",
    "yannick bolasie yala": "yannick bolasie",
    "romelu lukaku menama": "romelu lukaku",
    "pedro eliezer rodriguez ledesma": "pedro",
    "mario suarez mata": "mario suarez",
    "daniel william john ings": "danny ings",
    "jack frank porteous cork": "jack cork",
    "eunan o'kane" : "eunan o'kane",
    "Yann Kermorgant": "yann kermorgant"
}


In [48]:
sb_normalized = ["jose ramiro funes mori", "alexander banor tettey", "robert brady", "dame n''doye", "gary o''neil", "daniel andre sturridge", 
                 "roberto firmino barbosa de oliveira", "philippe coutinho correia", "adam david lallana", "bertrand isidore traore", 
                 "willian borges da silva", "francesc fabregas i soler", "kolo habib toure", "nathaniel edwin clyne", "james philip milner",
                 "jordan brian henderson", "bojan krkic perez", "ander herrera aguera", "marouane fellaini bakkioui", "juan manuel mata garcia", 
                 "juan carlos paredes reasco", "diego da silva costa", "oscar dos santos emboaba junior", "mame biram diouf", "guillermo varela olivera", 
                 "angelo obinze ogbonna", "alberto moreno perez", "divock okoth origi", "eunan o''kane", "sebastian coates nion", "andre ayew pele", 
                 "sung yeung ki", "gylfi or sigursson", "oriol romeu vidal", "jonathan howson", "ignacio monreal eraso", 
                 "joel nathaniel campbell samuels", "faustino marcos alberto rojo", "jesus navas gonzalez", "wayne mark rooney", "kurt happy zouma",
                 "cristian gamboa luna", "john o''shea", "mousa sidi yaya dembele", "jose miguel da rocha fonte", "radamel falcao garcia zarate",
                 "francis joseph coquelin", "gabriel imuetinyan agbonlahor", "max alain gradel", "jose manuel jurado marin", "chancel mbemba mangulu",
                 "dieumerci mbokani bezua", "sebastien aymar bassong nguena", "robert kenedy nunes do nascimento", "jonathan grant evans", 
                 "carles gil de pareja vicent", "brendan joel zibusiso galloway", "gerard deulofeu lazaro", "jose luis sanmartin mato", 
                 "clinton mua n''jie", "jefferson antonio montero vite", "fernando francisco reges", "david josue jimenez silva", 
                 "kelechi promise iheanacho", "andrew philip king", "lucas pezzini leiva", "cedric ricardo alves soares", "gabriel antoine obertan",
                 "adama traore diarra", "juan miguel jimenez lopez", "alexandre dimitri song billong", "luis antonio valencia mosquera", 
                 "victor chinedu anichebe", "marcin ryszard wasilewski", "pablo javier zabaleta girod", "martin gaston demichelis",
                 "gaston exequiel ramirez pereyra", "pedro mba obiang avomo", "wes morgan", "miguel angel britos cabrera", "angel rangel zaragoza",
                 "cesar azpilicueta tanco", "john michael nchekwube obinna", "rhu endly martina", "youssouf chafiq mulumbu ngangu",
                 "steven caulker", "sandro ranieri guimaraes cordeiro", "leroy fer", "daniel nii tackie mensah welbeck", "marc muniesa martinez",
                 "gilbert gianelli imbula wanga", "kevin linford stewart", "seydou doumbia", "jordi amat maas", "juan manuel iturbe arevalo", 
                 "osazemwinde peter odemwingie", "alexandre rodrigues da silva", "timothy evans fosu mensah", "emmanuel emenike", "connor steven randall",
                 "tammy bakumo abraham", "bradley shaun smith", "cheik ismael tiote", "tokelo anthony rantie", "matt grimes", "jose leonardo ulloa", 
                 "n''golo kante", "yann gerard m''vila", "jordi gomez garcia penche", "jose salomon rondon gimenez", "claudio ariel yacob", 
                 "idrissa gana gueye", "siem stefan de jong", "carlos alberto sanchez moreno", "ayoze perez gutierrez", "el hadji baye oumar niasse", 
                 "pape n''diaye souare", "odion jude ighalo", "chung yong lee", "gabriel armando de abreu", "alexis alejandro sanchez sanchez",
                 "hector bellerin moruno", "mohamed naser elsayed elneny", "santiago cazorla gonzalez", "oluwaseyi babajide ojo",  
                 "christian benteke liolo", "christian dannemann eriksen", "bamidele alli", "sergio leonel aguero del castillo", 
                 "nicolas hernan otamendi", "gnegneri yaya toure", "wilfried guemiand bony", "yannick bolasie yala", "romelu lukaku menama", 
                 "guangtai jiang", "enner remberto valencia lastra"]

fifa_normalized = [ "ramiro funes mori", "alexander tettey", "robbie brady", "dame n'doye", "gary o'neil", "daniel sturridge", "roberto firmino", 
                    "coutinho", "adam lallana", "bertrand traore", "willian", "cesc fabregas", "kolo toure", "nathaniel clyne", "james milner", 
                    "jordan henderson", "bojan", "ander herrera", "marouane fellaini", "juan mata", "juan carlos paredes", "diego costa", "oscar",
                    "mame diouf", "guillermo varela", "angelo ogbonna", "alberto moreno", "divock origi", "eunan o'kane", "sebastian coates", 
                    "andre ayew", "ki sung yueng", "gylfi sigursson", "oriol romeu", "jonny howson", "nacho monreal", "joel campbell", "marcos rojo", 
                    "jesus navas", "wayne rooney", "kurt zouma", "cristian gamboa", "john o'shea", "moussa dembele", "jose fonte", "falcao", 
                    "francis coquelin", "gabriel agbonlahor", "max gradel", "jurado", "chancel mbemba", "dieumerci mbokani", "sebastien bassong", 
                    "kenedy", "jonny evans", "carles gil", "brendan galloway", "deulofeu", "joselu", "clinton n'jie", "jefferson montero", "fernando", 
                    "david silva", "kelechi iheanacho", "andy king", "lucas leiva", "cedric", "gabriel obertan", "adama", "juanmi", "alexandre songl", 
                    "antonio valencia", "victor anichebe", "marcin wasilewski", "pablo zabaleta", "martin demichelis", "gaston ramirez", "pedro obiang", "wes morgan", "miguel angel britos", "angel rangel", "azpilicueta", "john obi mikel", "cuco martina", "youssouf mulumbu", 
                    "steven caulkerl", "sandrol", "leroy ferl", "danny welbeck", "marc muniesa", "giannelli imbula", "kevin stewart", "seydou doumbial", 
                    "jordi amat", "juan manuel iturbe", "peter odemwingie", "alexandre pato", "timothy fosu mensah", "emmanuel emenikel", 
                    "connor randall", "tammy abraham", "brad smith", "cheick tiote", "tokelo rantie", "matthew grimes", "leonardo ulloa" ,"n'golo kante",
                    "yann m'vila", "jordi gomez", "salomon rondon", "claudio yacob", "idrissa gueye", "siem de jong", "carlos sanchez", "ayoze perez", 
                    "oumar niasse", "pape souare", "odion ighalo", "lee chung yong", "gabriel", "alexis sanchez", "hector bellerin", "mohamed elneny",
                    "santi cazorla", "sheyi ojo", "christian benteke", "christian eriksen", "dele alli", "sergio aguero", "nicolas otamendi", "yaya toure",
                    "wilfried bony", "yannick bolasie", "romelu lukaku", "tyias browning", "enner valencia"]

# Names of players who player a portion of the EPL in 2015/2016: "Adam Johnson", "steven fletcher", "Éderzito António Macedo Lopes", "Bradley Johnson", 
# "Florian Thauvin", "Brede Hangeland", "yann kermorgant", "miguel arturo layun prado", "gary hooper", "modibo maiga", "juan guillermo cuadrado bello", 
# "Alessandro Diamanti", "steven taylor", "Mathieu Debuchy", "nikica jelavic"
# "
# wtfs: "3058, javier hernandez balcazar", "mikel arteta amatriain"

print(len(sb_normalized) - len(fifa_normalized))

for i in range(len(sb_normalized)):
    manual_overrides.update({sb_normalized[i]: fifa_normalized[i]})

print(manual_overrides)

0
{'fernando luiz rosa': 'fernandinho', 'fernando luiz roza': 'fernandinho', 'jose leonardo ulloa': 'leonardo ulloa', "n''golo kante": "n'golo kante", "n'golo kante": "n'golo kante", "yann gerard m''vila": "yann m'vila", "yann gerard m'vila": "yann m'vila", 'jordi gomez garcia penche': 'jordi gomez', 'jose salomon rondon gimenez': 'salomon rondon', 'claudio ariel yacob': 'claudio yacob', 'idrissa gana gueye': 'idrissa gueye', 'siem stefan de jong': 'siem de jong', 'carlos alberto sanchez moreno': 'carlos sanchez', 'ayoze perez gutierrez': 'ayoze perez', 'el hadji baye oumar niasse': 'oumar niasse', "pape n''diaye souare": 'pape souare', 'odion jude ighalo': 'odion ighalo', 'chung yong lee': 'lee chung yong', 'gabriel armando de abreu': 'gabriel', 'alexis alejandro sanchez sanchez': 'alexis sanchez', 'hector bellerin moruno': 'hector bellerin', 'mohamed naser elsayed elneny': 'mohamed elneny', 'santiago cazorla gonzalez': 'santi cazorla', 'oluwaseyi babajide ojo': 'sheyi ojo', 'christia

Building Manual Matches

In [49]:
manual_rows = []

for sb_norm, fifa_norm in manual_overrides.items():

    sb_match = sb_players[sb_players["statsbomb_name_norm"] == sb_norm]
    fifa_match = fifa_players[
        (fifa_players["fifa_name_norm"] == fifa_norm)
        | (fifa_players["match_name_norm"] == fifa_norm)
    ]

    if sb_match.empty:
        print(f"StatsBomb player not found: {sb_norm}")
        continue

    if fifa_match.empty:
        print(f"FIFA player not found: {fifa_norm}")
        continue

    for _, sb_row in sb_match.iterrows():
        fifa_row = fifa_match.iloc[0]

        manual_rows.append({
            "player_id": sb_row["player_id"],
            "player": sb_row["player"],
            "statsbomb_name_norm": sb_row["statsbomb_name_norm"],
            "Player Name": fifa_row["Player Name"],
            "fifa_name_norm": fifa_row["fifa_name_norm"],
            "match_name_norm": fifa_row["match_name_norm"],
            "Att. Position": fifa_row["Att. Position"],
            "Finishing": fifa_row["Finishing"],
            "Shot Power": fifa_row["Shot Power"],
            "Long Shots": fifa_row["Long Shots"],
            "Volleys": fifa_row["Volleys"],
            "Penalties": fifa_row["Penalties"],
            "Heading": fifa_row["Heading"],
            "Source": fifa_row["Source"],
            "match_method": "manual",
            "match_score": 100
        })

manual_matches = pd.DataFrame(manual_rows)


FIFA player not found: fernandinho
FIFA player not found: fernandinho
FIFA player not found: leonardo ulloa
FIFA player not found: n'golo kante
FIFA player not found: n'golo kante
FIFA player not found: yann m'vila
FIFA player not found: yann m'vila
FIFA player not found: jordi gomez
FIFA player not found: salomon rondon
FIFA player not found: claudio yacob
FIFA player not found: idrissa gueye
FIFA player not found: siem de jong
FIFA player not found: carlos sanchez
FIFA player not found: ayoze perez
FIFA player not found: oumar niasse
FIFA player not found: pape souare
FIFA player not found: odion ighalo
FIFA player not found: lee chung yong
FIFA player not found: gabriel
FIFA player not found: alexis sanchez
FIFA player not found: hector bellerin
FIFA player not found: mohamed elneny
FIFA player not found: santi cazorla
FIFA player not found: sheyi ojo
FIFA player not found: christian benteke
FIFA player not found: christian eriksen
FIFA player not found: dele alli
FIFA player not fo

In [ ]:
# exists = 'Mario Suárez Mata' in shots['player'].values

# print(exists)

Standardizing Exact Matches

In [50]:
exact_mapping = exact_matches[[
    "player_id",
    "player",
    "statsbomb_name_norm",
    "Player Name",
    "fifa_name_norm",
    "match_name_norm",
    "Att. Position", 
    "Finishing", 
    "Shot Power", 
    "Long Shots", 
    "Volleys", 
    "Penalties", 
    "Heading",
    "Source",
    "match_method",
    "match_score"
]].copy()


Combine Final Mapping Table

In [51]:
player_abilities = pd.concat(
    [exact_mapping, manual_matches],
    ignore_index=True
)

player_abilities = player_abilities.drop_duplicates(
    subset=["player_id"],
    keep="last"
)

player_abilities.head()

,player_id,player,statsbomb_name_norm,Player Name,fifa_name_norm,match_name_norm,Att. Position,Finishing,Shot Power,Long Shots,Volleys,Penalties,Heading,Source,match_method,match_score
0,28100.0,Ramires Santos do Nascimento,ramires santos do nascimento,Ramires Santos do Nascimen.,ramires santos do nascimen,ramires santos do nascimento,77.0,68.0,77.0,72.0,75.0,85.0,64.0,manual_fifa_index,exact,100
1,9554.0,Steven Fletcher,steven fletcher,Steven Fletcher,steven fletcher,steven fletcher,78.0,80.0,76.0,67.0,66.0,71.0,86.0,manual_fifa_index,exact,100
2,15996.0,Éderzito António Macedo Lopes,ederzito antonio macedo lopes,Éder Macedo Lopes,eder macedo lopes,ederzito antonio macedo lopes,78.0,73.0,79.0,69.0,75.0,73.0,7.0,manual_fifa_index,exact,100
3,42703.0,Mikel Arteta Amatriain,mikel arteta amatriain,Mikel Arteta Amatriain,mikel arteta amatriain,mikel arteta amatriain,70.0,75.0,74.0,78.0,67.0,68.0,44.0,manual_fifa_index,exact,100
4,42902.0,Adam Johnson,adam johnson,Adam Johnson,adam johnson,adam johnson,77.0,75.0,74.0,78.0,67.0,68.0,44.0,manual_fifa_index,exact,100


These players played a portion of the season and were transferred. They are hence not present in the FIFA16 dataset for players in the EPL. See the comments in the manual overide cell.

Now refer back to with commented supplemental_players, uncomment, and rerun the pipeline.

In [52]:
player_abilities.head(100)
# df_extended.to_csv("/Users/nanakwamekankam/Desktop/Football_Stats/xG/extended_fifa16.csv")

,player_id,player,statsbomb_name_norm,Player Name,fifa_name_norm,match_name_norm,Att. Position,Finishing,Shot Power,Long Shots,Volleys,Penalties,Heading,Source,match_method,match_score
0,28100.0,Ramires Santos do Nascimento,ramires santos do nascimento,Ramires Santos do Nascimen.,ramires santos do nascimen,ramires santos do nascimento,77.0,68.0,77.0,72.0,75.0,85.0,64.0,manual_fifa_index,exact,100
1,9554.0,Steven Fletcher,steven fletcher,Steven Fletcher,steven fletcher,steven fletcher,78.0,80.0,76.0,67.0,66.0,71.0,86.0,manual_fifa_index,exact,100
2,15996.0,Éderzito António Macedo Lopes,ederzito antonio macedo lopes,Éder Macedo Lopes,eder macedo lopes,ederzito antonio macedo lopes,78.0,73.0,79.0,69.0,75.0,73.0,7.0,manual_fifa_index,exact,100
3,42703.0,Mikel Arteta Amatriain,mikel arteta amatriain,Mikel Arteta Amatriain,mikel arteta amatriain,mikel arteta amatriain,70.0,75.0,74.0,78.0,67.0,68.0,44.0,manual_fifa_index,exact,100
4,42902.0,Adam Johnson,adam johnson,Adam Johnson,adam johnson,adam johnson,77.0,75.0,74.0,78.0,67.0,68.0,44.0,manual_fifa_index,exact,100
5,27853.0,Mauro Matías Zárate,mauro matias zarate,Mauro Zárate,mauro zarate,mauro matias zarate,79.0,75.0,80.0,82.0,78.0,70.0,58.0,manual_fifa_index,exact,100
6,4804.0,Bradley Johnson,bradley johnson,Bradley Johnson,bradley johnson,bradley johnson,72.0,70.0,91.0,78.0,68.0,72.0,76.0,manual_fifa_index,exact,100
7,3254.0,Florian Thauvin,florian thauvin,Florian Thauvin,florian thauvin,florian thauvin,74.0,70.0,76.0,74.0,64.0,57.0,66.0,manual_fifa_index,exact,100
8,20543.0,Nikica Jelavić,nikica jelavic,Nikica Jelavić,nikica jelavic,nikica jelavic,75.0,77.0,74.0,70.0,69.0,81.0,77.0,manual_fifa_index,exact,100
9,43695.0,Brede Hangeland,brede hangeland,Brede Hangeland,brede hangeland,brede hangeland,19.0,34.0,63.0,39.0,38.0,44.0,80.0,manual_fifa_index,exact,100


Players like Adam Johnson, Steven Fletcher, Der, Bradley Johnson etc, played in different leagues during the 2015/2016 EPL season due to transfers and loans, so they aren't featured in the dataset(which is a subset for the fifa16 version). However, they do have ratings for fifa16, so I will manually add those to the dataset.

## SAVING DATASET

In [ ]:
player_abilities.to_csv("/Users/nanakwamekankam/Desktop/Football_Stats/xG/datasets/processed/player_ratings.csv")

To get the goalkeeper ability, FIFA avaerges the features for goalkeeper. I will do this to generate a saving column but will not include columns like goalkeeper_kicking